# Organized clonotype analysis notebook

This notebook was reorganized by adding markdown headers only. The original code cells were preserved unchanged and kept in their original order.

## Table of contents

1. Setup and imports
2. Data loading and metadata preparation
3. Initial visualization and filtering
4. TCR chain indexing and quality control
5. AIRR filtering
6. Clonotype definition and clonotype network
7. TCRdist clonotype clusters
8. CDR3 extraction and top clonotype inspection
9. V-gene-aware clonotype definition
10. Clonal expansion analysis
11. Differential expression in selected clonotypes
12. Gene usage analysis


## TCR analysis

# 1. Setup and imports

Libraries used for MuData, scirpy, plotting, and downstream analysis.


In [ ]:
# =========================
# Standard library
# =========================
import os
from pathlib import Path
from functools import partial
from multiprocessing import Pool
from typing import Optional

# =========================
# Core scientific computing
# =========================
import numpy as np
import pandas as pd
import scipy.sparse as sp
from scipy.sparse import csr_matrix
from scipy.stats import median_abs_deviation

# =========================
# Single-cell analysis
# =========================
import anndata as ad
from anndata import AnnData

import scanpy as sc
import scirpy as ir
#import muon as mu
#from muon import MuData
#import mudata as mu
import muon as mu
import palantir
import decoupler as dc

# =========================
# Differential expression
# =========================
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats

try:
    # newer pydeseq2
    from pydeseq2.default_inference import DefaultInference
except Exception:
    # older pydeseq2
    from pydeseq2.dds import DefaultInference

# =========================
# Visualization
# =========================
import matplotlib.pyplot as plt
from matplotlib import rcParams
from matplotlib.pyplot import rc_context
from matplotlib import cm as mpl_cm
from cycler import cycler
from matplotlib_venn import venn3

import seaborn as sns
import altair as alt

# Enable Altair backend
alt.data_transformers.enable("vegafusion")


## Import data

# 2. Data loading and metadata preparation

Load the MuData object and harmonize sample, treatment, condition, and annotation metadata.


In [ ]:
mdata = mu.read_h5mu("/data/projects/2021/MicrobialMetabolites/single-cell-sorted-cd8/results/40_gex_surface_prot/002_annotate_mudata.h5mu")

In [ ]:
rename_dict = {
    "10mix_ICI1": "ctrl1",
    "10mix_ICI2": "ctrl2",
    "11mix_ICI1": "effector1",
    "11mix_ICI2": "effector2",
    "GF_ICI1": "GF_noICI1",
    "GF_ICI2": "GF_noICI2",
    "GF_ICI1_plus": "GF1",
    "GF_ICI2_plus": "GF2"
}

mdata["gex"].obs["sample_id"] = (
    mdata["gex"].obs["sample_id"]
    .replace(rename_dict)
)

In [ ]:
mdata["airr"].obs["sample_id"] = (
    mdata["gex"].obs["sample_id"]
    .replace(rename_dict)
)

In [ ]:
mdata["airr"].obs["sample_id"]

In [ ]:
mdata["airr"].obs["group4"] =    mdata["gex"].obs["group4"] 

In [ ]:
mdata.update()

# 3. Initial visualization and filtering

Inspect annotations on UMAP and remove unwanted cell populations.


In [ ]:
sc.pl.umap(mdata["gex"], color=["cell_annotation_05","group4","sample_id"], frameon=False)

## Creating chain indices & TCR Quality Control

# 4. TCR chain indexing and quality control

Index AIRR chains and evaluate receptor subtype / chain pairing quality.


In [ ]:
ir.pp.index_chains(mdata)
ir.tl.chain_qc(mdata)

In [ ]:
fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(10, 4), gridspec_kw={"wspace": 0.5})
mu.pl.embedding(mdata, basis="gex:umap", color=["Cd3e"], ax=ax0, show=False, frameon=False, cmap= "viridis", vmax= "p99")
mu.pl.embedding(mdata, basis="gex:umap", color=["airr:receptor_type"], ax=ax1, frameon=False)

In [ ]:
_ = ir.pl.group_abundance(mdata, groupby="airr:receptor_subtype", target_col="gex:cell_annotation_05")

In [ ]:
_ = ir.pl.group_abundance(mdata, groupby="airr:chain_pairing", target_col="gex:cell_annotation_05")

In [ ]:
print(
    "Fraction of cells with more than one pair of TCRs: {:.2f}".format(
        np.sum(mdata.obs["airr:chain_pairing"].isin(["extra VJ", "extra VDJ", "two full chains", "multichain"]))
        / mdata["airr"].n_obs
    )
)

# 5. AIRR filtering

Filter cells based on TCR chain pairing quality before clonotype analysis.


In [ ]:
mu.pp.filter_obs(mdata, "airr:chain_pairing", lambda x: ~np.isin(x, ["orphan VDJ", "orphan VJ"]))

In [ ]:
ax = ir.pl.group_abundance(mdata, groupby="airr:chain_pairing", target_col="gex:group4")

In [ ]:
ax = ir.pl.group_abundance(mdata, groupby="airr:receptor_subtype", target_col="gex:group4")

In [ ]:
ax = ir.pl.group_abundance(mdata, groupby="airr:receptor_type", target_col="gex:group4")

## After filtering

In [ ]:
# cells in AIRR that are TCR
tcr_cells = mdata["airr"].obs_names[
    mdata["airr"].obs["receptor_type"] == "TCR"
]

# keep only those cells in the full MuData object
mdata = mdata[tcr_cells, :].copy()

In [ ]:
mdata.update()

In [ ]:
mdata["airr"].obs["receptor_type"].value_counts()

In [ ]:
for key in [
    "airr:receptor_type_colors",
    "receptor_type_colors",
]:
    if key in mdata.uns:
        del mdata.uns[key]

In [ ]:
mdata["airr"].obs["receptor_type"] = (
    mdata["airr"].obs["receptor_type"]
    .astype("category")
    .cat.remove_unused_categories()
)

In [ ]:
mdata.obs["receptor_type"] = mdata["airr"].obs["receptor_type"].reindex(mdata.obs_names)

mdata.obs["receptor_type"] = (
    mdata.obs["receptor_type"]
    .astype("category")
    .cat.remove_unused_categories()
)

if "receptor_type_colors" in mdata.uns:
    del mdata.uns["receptor_type_colors"]

mu.pl.embedding(
    mdata,
    basis="gex:umap",
    color=["Cd3e","receptor_type"],cmap= "viridis", vmax= "p99",
    frameon=False,
)

## Define clonotypes and clonotype clusters

### Compute CDR3 neighborhood graph and define clonotypes

# 6. Clonotype definition and clonotype network

Define clonotypes from CDR3 sequence similarity and visualize networks by metadata.


In [ ]:
mdata = mdata_filtered

In [ ]:
# using default parameters, `ir_dist` will compute nucleotide sequence identity
ir.pp.ir_dist(mdata)
ir.tl.define_clonotypes(mdata, receptor_arms="all", dual_ir="primary_only")

In [ ]:
ir.tl.clonotype_network(mdata, min_cells=500)

In [ ]:
mdata.obs.groupby("gex:cell_annotation_05", dropna=False).size()

In [ ]:
_ = ir.pl.clonotype_network(mdata, color="gex:cell_annotation_05", base_size=1, label_fontsize=9, panel_size=(7, 7))

In [ ]:
_ = ir.pl.clonotype_network(mdata, color="gex:group4", base_size=1, label_fontsize=9, panel_size=(7, 7))

In [ ]:
_ = ir.pl.clonotype_network(mdata, color="gex:sample_id", base_size=1, label_fontsize=9, panel_size=(7, 7))

### Re-compute CDR3 neighborhood graph and define clonotype clusters

# 7. TCRdist clonotype clusters

Recompute CDR3 distances using TCRdist and define clonotype clusters.


In [ ]:
ir.pp.ir_dist(
    mdata,
    metric="tcrdist",
    sequence="aa",
    cutoff=15,
)

In [ ]:
ir.tl.define_clonotype_clusters(mdata, sequence="aa", metric="tcrdist", receptor_arms="all", dual_ir="any")

In [ ]:
ir.tl.clonotype_network(mdata, min_cells=500, sequence="aa", metric="tcrdist")

In [ ]:
_ = ir.pl.clonotype_network(mdata, color="gex:cell_annotation_05", label_fontsize=9, panel_size=(7, 7), base_size=1, frameon=False)

In [ ]:
_ = ir.pl.clonotype_network(mdata, color="gex:group4", label_fontsize=9, panel_size=(7, 7), base_size=1, frameon=False)

In [ ]:
_ = ir.pl.clonotype_network(mdata, color="gex:sample_id", label_fontsize=9, panel_size=(7, 7), base_size=1, frameon=False)

In [ ]:
ir.tl.define_clonotype_clusters(mdata, sequence="aa", metric="tcrdist", receptor_arms="all", dual_ir="any")

# 8. CDR3 extraction and top clonotype inspection

Extract CDR3 information and inspect selected clonotype clusters.

Comple clusters are the ones having info for all the regions


In [ ]:
cols = [
    "VJ_1_junction_aa",
    "VDJ_1_junction_aa",
    "VJ_2_junction_aa",
    "VDJ_2_junction_aa",
]

with ir.get.airr_context(mdata, "junction_aa", ["VJ_1", "VDJ_1", "VJ_2", "VDJ_2"]):

    cdr3_complete = (
        mdata.obs
        # keep only cells where none of the CDR3 columns are NaN
        .loc[lambda x: x[cols].notna().all(axis=1)]
        .groupby(
            cols + [
                "airr:cc_aa_tcrdist",
                "airr:receptor_subtype",
            ],
            observed=True,
            dropna=False,
        )
        .size()
        .reset_index(name="n_cells")
        .sort_values("n_cells", ascending=False)
    )



In [ ]:
cdr3_complete

In [ ]:
# count abundance per clonotype
clone_counts = (
    mdata.obs["airr:clone_id"]
    .value_counts()
    .rename_axis("airr:clone_id")
    .reset_index(name="n_cells")
)

# take top N (same as max_cols=60)
top_clones = clone_counts.head(60)["airr:clone_id"]

In [ ]:
top_clones.head()

In [ ]:
complete_clusters = cdr3_complete["airr:cc_aa_tcrdist"].unique()

In [ ]:
complete_clusters

In [ ]:
clone_to_cluster = (
    mdata.obs[["airr:clone_id", "airr:cc_aa_tcrdist"]]
    .drop_duplicates()
)

In [ ]:
top_clone_clusters = clone_to_cluster[
    clone_to_cluster["airr:clone_id"].isin(top_clones)
]["airr:cc_aa_tcrdist"]

# intersection
overlap = set(top_clone_clusters) & set(complete_clusters)

overlap

In [ ]:
top_complete = clone_to_cluster[
    (clone_to_cluster["airr:clone_id"].isin(top_clones)) &
    (clone_to_cluster["airr:cc_aa_tcrdist"].isin(complete_clusters))
]

top_complete

## Top clones

In [ ]:
_ = ir.pl.group_abundance(mdata, groupby="airr:clone_id", target_col="gex:group4", max_cols=60, figsize=(20, 3))

## Top clonotypes filtered by threshold  > 100 cells 

In [ ]:
import pandas as pd

clone_sizes = (
    mdata.obs["airr:clone_id"]
    .value_counts()
    .rename_axis("clone_id")
    .reset_index(name="n_cells")
)

In [ ]:
large_clones = clone_sizes.loc[clone_sizes["n_cells"] > 300]

In [ ]:
abundance_df = pd.crosstab(
    mdata.obs["airr:clone_id"],
    mdata.obs["gex:group4"]
)

In [ ]:
clone_counts = abundance_df.sum(axis=1)

abundance_df_filtered = abundance_df.loc[clone_counts > 300]

In [ ]:
top_clone_ids = abundance_df_filtered.index.to_list()

In [ ]:
top_clone_ids

In [ ]:
import scanpy as sc

keep_clones = abundance_df_filtered.index

mdata_large = mdata[
    mdata.obs["airr:clone_id"].isin(keep_clones)
].copy()

In [ ]:
mdata_large

In [ ]:
ir.pl.group_abundance(
    mdata_large,
    groupby="airr:clone_id",
    target_col="gex:group4",
    figsize=(20, 5)
)

## Explore specific clonotypes

### 211

In [ ]:
with ir.get.airr_context(mdata, "junction_aa", ["VJ_1", "VDJ_1", "VJ_2", "VDJ_2"]):
    cdr3_ct_211 = (
        # TODO astype(str) is required due to a bug in pandas ignoring `dropna=False`. It seems fixed in pandas 2.x
        mdata.obs.loc[lambda x: x["airr:cc_aa_tcrdist"] == "211"]
        .astype(str)
        .groupby(
            [
                "VJ_1_junction_aa",
                "VDJ_1_junction_aa",
                "VJ_2_junction_aa",
                "VDJ_2_junction_aa",
                "airr:receptor_subtype",
            ],
            observed=True,
            dropna=False,
        )
        .size()
        .reset_index(name="n_cells")
    )
cdr3_ct_211

### 5967

In [ ]:
with ir.get.airr_context(mdata, "junction_aa", ["VJ_1", "VDJ_1", "VJ_2", "VDJ_2"]):
    cdr3_ct_5967 = (
        # TODO astype(str) is required due to a bug in pandas ignoring `dropna=False`. It seems fixed in pandas 2.x
        mdata.obs.loc[lambda x: x["airr:cc_aa_tcrdist"] == "5967"]
        .astype(str)
        .groupby(
            [
                "VJ_1_junction_aa",
                "VDJ_1_junction_aa",
                "VJ_2_junction_aa",
                "VDJ_2_junction_aa",
                "airr:receptor_subtype",
            ],
            observed=True,
            dropna=False,
        )
        .size()
        .reset_index(name="n_cells")
    )
cdr3_ct_5967

### Including the V-gene in clonotype definition

# 9. V-gene-aware clonotype definition

Include V-gene usage in clonotype definitions and inspect clusters with variable V-gene assignments.


In [ ]:
ir.tl.define_clonotype_clusters(
    mdata,
    sequence="aa",
    metric="tcrdist",
    receptor_arms="all",
    dual_ir="any",
    same_v_gene=True,
    key_added="cc_aa_tcrdist_same_v",
)

In [ ]:
# find clonotypes with more than one `clonotype_same_v`
ct_different_v = mdata.obs.groupby("airr:cc_aa_tcrdist").apply(lambda x: x["airr:cc_aa_tcrdist_same_v"].nunique() > 1)
ct_different_v = ct_different_v[ct_different_v].index.values.tolist()
#ct_different_v

In [ ]:
with ir.get.airr_context(mdata, "v_call", ["VJ_1", "VDJ_1","VJ_2","VDJ_2"]):
    ct_different_v_df = (
        mdata.obs.loc[
            lambda x: x["airr:cc_aa_tcrdist"].isin(ct_different_v),
            [
                "airr:cc_aa_tcrdist",
                "airr:cc_aa_tcrdist_same_v",
                "VJ_1_v_call",
                "VDJ_1_v_call"
            ],
        ]
        .sort_values("airr:cc_aa_tcrdist")
        .drop_duplicates()
        .reset_index(drop=True)
    )
ct_different_v_df

In [ ]:
ct_different_v_df["airr:cc_aa_tcrdist"].unique()

In [ ]:
ct_different_v_df_211 = ct_different_v_df.loc[
    ct_different_v_df["airr:cc_aa_tcrdist"].astype(str) == "211"
]

In [ ]:
ct_different_v_df_18 = ct_different_v_df.loc[
    ct_different_v_df["airr:cc_aa_tcrdist"].astype(str) == "18"
]

In [ ]:
with ir.get.airr_context(
    mdata,
    ["v_call", "j_call"],
    ["VJ_1", "VDJ_1"]
):
    ct_different_j_df = (
        mdata.obs.loc[:, [
            "airr:cc_aa_tcrdist",
            "airr:cc_aa_tcrdist_same_v",
            "VJ_1_v_call",
            "VDJ_1_v_call",
            "VJ_1_j_call",
            "VDJ_1_j_call",
        ]]
        .dropna(subset=[
            "VJ_1_j_call",
            "VDJ_1_j_call",
        ])
        .drop_duplicates()
    )

ct_different_j_df_211 = ct_different_j_df.loc[
    ct_different_j_df["airr:cc_aa_tcrdist"].astype(str) == "211"
]

ct_different_j_df_211

In [ ]:
with ir.get.airr_context(
    mdata,
    ["v_call", "j_call"],
    ["VJ_1", "VDJ_1"]
):
    ct_different_j_df = (
        mdata.obs.loc[:, [
            "airr:cc_aa_tcrdist",
            "airr:cc_aa_tcrdist_same_v",
            "VJ_1_v_call",
            "VDJ_1_v_call",
            "VJ_1_j_call",
            "VDJ_1_j_call",
        ]]
        .dropna(subset=[
            "VJ_1_j_call",
            "VDJ_1_j_call",
        ])
        .drop_duplicates()
    )

ct_different_j_df_18 = ct_different_j_df.loc[
    ct_different_j_df["airr:cc_aa_tcrdist"].astype(str) == "18"
]

ct_different_j_df_18

## Clonotype analysis

### Clonal expansion

# 10. Clonal expansion analysis

Quantify and visualize clonal expansion across samples, groups, and annotations.


In [ ]:
ir.tl.clonal_expansion(mdata, breakpoints=(1, 500,1000))

In [ ]:
mdata["airr"].obs["clonal_expansion"]

In [ ]:
# keep AIRR cells with non-missing clonal_expansion
keep_cells = mdata["airr"].obs_names[
    mdata["airr"].obs["clonal_expansion"].notna()
]

# subset full MuData by shared cell names
mdata_filtered = mdata[keep_cells, :].copy()

In [ ]:
# subset cells with non-missing clonal_expansion
keep_cells = mdata["airr"].obs_names[
    mdata["airr"].obs["clonal_expansion"].notna()
]
keep_cells = keep_cells.intersection(mdata["gex"].obs_names)

mdata_filtered = mdata[keep_cells, :].copy()

# clean unused categories
mdata_filtered["airr"].obs["clonal_expansion"] = (
    mdata_filtered["airr"].obs["clonal_expansion"]
    .cat.remove_unused_categories()
)

# remove old stored colors if present
if "clonal_expansion_colors" in mdata_filtered["airr"].uns:
    del mdata_filtered["airr"].uns["clonal_expansion_colors"]

In [ ]:
mu.pl.embedding(
    mdata_filtered,
    basis="gex:umap",
    color=["airr:clonal_expansion"],
    frameon=False
)
#plt.savefig("gex_umap_clonal_expansion.png", dpi=300, bbox_inches="tight")
plt.close()

In [ ]:
if "airr:clonal_expansion_colors" in mdata_filtered.uns:
    del mdata_filtered.uns["airr:clonal_expansion_colors"]

In [ ]:
mu.pl.embedding(mdata_filtered, basis="gex:umap", color=["airr:clonal_expansion", "airr:clone_id_size"], frameon=False, )

In [ ]:
sc.pl.umap(mdata["gex"], color=["sample_id"], groups = ["GF_noICI2"], frameon=False)

In [ ]:
sc.pl.umap(mdata["gex"], color=["sample_id"], groups = ["GF_noICI1"], frameon=False)

In [ ]:
sc.pl.umap(mdata["gex"], color=["sample_id"], groups = ["GF1"], frameon=False)

In [ ]:
sc.pl.umap(mdata["gex"], color=["sample_id"], groups = ["GF2"], frameon=False)

In [ ]:
sc.pl.umap(mdata["gex"], color=["sample_id"], groups = ["ctrl1"], frameon=False)

In [ ]:
sc.pl.umap(mdata["gex"], color=["sample_id"], groups = ["ctrl2"], frameon=False)

In [ ]:
sc.pl.umap(mdata["gex"], color=["sample_id"], groups = ["effector1"], frameon=False)

In [ ]:
sc.pl.umap(mdata["gex"], color=["sample_id"], groups = ["effector2"], frameon=False)

In [ ]:
import pandas as pd

mdata.obs["gex:group4"] = pd.Categorical(
    mdata.obs["gex:group4"],
    categories=["GF_noici", "GF", "ctrl", "effector"],
    ordered=True
)

In [ ]:
_ = ir.pl.clonal_expansion(mdata, target_col="clone_id", groupby="gex:group4", breakpoints=(1, 500,1000), normalize=True)

In [ ]:
import matplotlib.pyplot as plt
import os

# create output directory
os.makedirs("./figures", exist_ok=True)

# generate plot
_ = ir.pl.clonal_expansion(
    mdata,
    target_col="clone_id",
    groupby="gex:group4",
    breakpoints=(1, 500, 1000),
    normalize=True
)

# get current axes and figure
ax = plt.gca()
fig = plt.gcf()

# axis labels and title
ax.set_ylabel("Number of cells")
ax.set_xlabel("")

# save
#fig.savefig("./figures/clonal_expansion.png", dpi=300, bbox_inches="tight")
#fig.savefig("./figures/clonal_expansion.svg", bbox_inches="tight")

plt.show()

In [ ]:
df = ir.tl.clonal_expansion(
    mdata,
    target_col="clone_id",
    expanded_in="gex:group4",
    breakpoints=(1, 500, 1000),

    inplace=False
)

In [ ]:
import pandas as pd

# convert index to series
cells = df.index.to_series()

group4 = pd.Series(index=cells.index, dtype="object")

group4[cells.str.contains("GF-ICI1")] = "GF_noICI"
group4[cells.str.contains("GF-ICI2")] = "GF_noICI"
group4[cells.str.contains("GF-ICI1-plus")] = "GF_ICI"  
group4[cells.str.contains("GF-ICI2-plus")] = "GF_ICI"   # adjust if needed
group4[cells.str.contains("10mix")] = "ctrl_ICI"
group4[cells.str.contains("11mix")] = "effector_ICI"

In [ ]:
df_combined = pd.DataFrame({
    "group4": group4,
    "clonal_expansion": df
})

In [ ]:
counts = pd.crosstab(
    df_combined["group4"],
    df_combined["clonal_expansion"]
)

counts

## Convergent evolution

Receptors that (likely) recognize the same antigen but have evolved from different clones.

In [ ]:
mdata["airr"]

In [ ]:
mdata["gex"].obs["group4"]

In [ ]:
ir.tl.clonotype_convergence(mdata, key_coarse="cc_aa_tcrdist", key_fine="clone_id")

In [ ]:
mu.pl.embedding(mdata, "gex:umap", color=["airr:is_convergent"], groups=["convergent"],frameon=False)

# 11. Differential expression in selected clonotypes

Subset selected clonotypes and compare gene expression programs across groups.


In [ ]:
clones_cells = (
    mdata["airr"].obs_names[
        mdata["airr"].obs["clone_id"].astype(str).isin(top_clone_ids)
    ]
)

clones_df = mdata["gex"].obs.loc[
    mdata["gex"].obs_names.intersection(clones_cells)
]

clones_df["group4"].value_counts(normalize=True)

In [ ]:
pd.crosstab(
    clones_df["group4"],
    clones_df["cell_annotation_05"],
    normalize="index"
)

In [ ]:
clones_gex = mdata["gex"][
    mdata["gex"].obs_names.intersection(clones_cells)
].copy()

In [ ]:
sc.tl.rank_genes_groups(
    clones_gex,
    groupby="group4",
    groups=["effector"],
    reference="ctrl",
    method="wilcoxon"
)

In [ ]:
sc.pl.rank_genes_groups(
    clones_gex,
    n_genes=20
)

In [ ]:
deg_df = sc.get.rank_genes_groups_df(
    clones_gex,
    group=None   # all groups
)

# View top rows
deg_df.head(20)

In [ ]:
from adjustText import adjust_text

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# for arrows / better labels
from adjustText import adjust_text

# Example dataframe
df = deg_df.copy()

df["neg_log10_padj"] = -np.log10(df["pvals_adj"])

# thresholds
lfc_cutoff = 1
padj_cutoff = 0.01

df["significant"] = (
    (df["pvals_adj"] < padj_cutoff) &
    (abs(df["logfoldchanges"]) > lfc_cutoff)
)

# plot
plt.figure(figsize=(10, 10))

# all genes
plt.scatter(
    df["logfoldchanges"],
    df["neg_log10_padj"],
    s=12,
    alpha=0.5
)

# significant genes
sig_df = df[df["significant"]]

plt.scatter(
    sig_df["logfoldchanges"],
    sig_df["neg_log10_padj"],
    s=20,
    alpha=0.9
)

# add labels
texts = []

for _, row in sig_df.iterrows():
    texts.append(
        plt.text(
            row["logfoldchanges"],
            row["neg_log10_padj"],
            row["names"],
            fontsize=13
        )
    )

# add arrows and avoid overlap
adjust_text(
    texts,
    arrowprops=dict(
        arrowstyle="->",
        color="black",
        lw=1
    )
)

# threshold lines
plt.axvline(lfc_cutoff, linestyle="--")
plt.axvline(-lfc_cutoff, linestyle="--")
plt.axhline(-np.log10(padj_cutoff), linestyle="--")

plt.xlabel("log2 fold change")
plt.ylabel("-log10 adjusted p-value")
plt.title("Volcano plot")

plt.tight_layout()
#plt.savefig("volcano_plot_deg_top60clonotypes.svg")
plt.show()

In [ ]:
#deg_df.to_csv("top60_clonotypes_deg_genes.csv")

In [ ]:
# Subset significant genes
df_significant = df[df["significant"]].copy()

# View
print(df_significant.head())

# Save as CSV
#df_significant.to_csv("top60_clonotypes_sig_deg_genes.csv", index=False)

## Gene usage

# 12. Gene usage analysis

Analyze V/J gene usage across clonotypes and experimental groups.


In [ ]:
import pandas as pd

mdata.obs["gex:sample_id"] = pd.Categorical(
    mdata.obs["gex:sample_id"],
    categories=["GF_noICI1", "GF_noICI2", "GF1","GF2","ctrl1", "ctrl2","effector1","effector2"],
    ordered=True
)

In [ ]:
mdata.update()

In [ ]:
mdata

In [ ]:
import matplotlib.pyplot as plt

with ir.get.airr_context(mdata, "v_call"):
    ir.pl.group_abundance(
        mdata,
        groupby="VJ_1_v_call",
        target_col="gex:group4",
        normalize=True,
        max_cols=20,
        figsize=(24, 6)   # wider figure
    )

plt.xticks(rotation=45, ha="right", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

with ir.get.airr_context(mdata, "v_call"):
    ir.pl.group_abundance(
        mdata,
        groupby="VJ_1_v_call",
        target_col="gex:cell_annotation_05",
        normalize=True,
        max_cols=20,
        figsize=(24, 6)   # wider figure
    )

plt.xticks(rotation=45, ha="right", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
with ir.get.airr_context(mdata, "v_call"):

    # Create the same table used for the plot
    df_abundance = (
        mdata.obs
        .groupby(["VDJ_1_v_call", "gex:group4"])
        .size()
        .unstack(fill_value=0)
    )

    # Optional normalization (same as normalize=True in the plot)
    df_abundance = df_abundance.div(df_abundance.sum(axis=0), axis=1)

# View dataframe
print(df_abundance)

# Top 20 rows by total abundance (same logic as max_cols=20)
top_df = (
    df_abundance.loc[df_abundance.sum(axis=1)
    .sort_values(ascending=False)
    #.head(20)
    .index]
)

#top_df.to_csv("top_abundant_genes_VDJ_1_v_call.csv")

In [ ]:
with ir.get.airr_context(mdata, "v_call"):
    ir.pl.group_abundance(
        mdata[
            mdata.obs["VJ_1_v_call"].isin(["TRAV12-3","TRAV6N-7"]),
            :,
        ],
        groupby="gex:group4",
        target_col="VJ_1_v_call",
        normalize=False,
    )

In [ ]:
import matplotlib.pyplot as plt

with ir.get.airr_context(mdata, "v_call"):
    ir.pl.group_abundance(
        mdata,
        groupby="VJ_2_v_call",
        target_col="gex:group4",
        normalize=True,
        max_cols=20,
        figsize=(24, 6)   # wider figure
    )

plt.xticks(rotation=45, ha="right", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
with ir.get.airr_context(mdata, "v_call"):

    # Create the same table used for the plot
    df_abundance = (
        mdata.obs
        .groupby(["VDJ_2_v_call", "gex:group4"])
        .size()
        .unstack(fill_value=0)
    )

    # Optional normalization (same as normalize=True in the plot)
    df_abundance = df_abundance.div(df_abundance.sum(axis=0), axis=1)

# View dataframe
print(df_abundance)

# Top 20 rows by total abundance (same logic as max_cols=20)
top_df = (
    df_abundance.loc[df_abundance.sum(axis=1)
    .sort_values(ascending=False)
    #.head(20)
    .index]
)


#top_df.to_csv("top_abundant_genes_VDJ_2_v_call.csv")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

sample_order = [
    "GF_noICI1", "GF_noICI2",
    "GF1", "GF2",
    "ctrl1", "ctrl2",
    "effector1", "effector2"
]

with ir.get.airr_context(mdata, "v_call"):

    obs_sub = mdata.obs.loc[
        mdata.obs["VJ_2_v_call"].isin(["TRAV10", "TRAV12-1"]),
        ["gex:sample_id", "VJ_2_v_call"]
    ].copy()

obs_sub["gex:sample_id"] = pd.Categorical(
    obs_sub["gex:sample_id"],
    categories=sample_order,
    ordered=True
)

abundance_df = pd.crosstab(
    obs_sub["gex:sample_id"],
    obs_sub["VJ_2_v_call"],
    normalize="index"
)

abundance_df = abundance_df.reindex(sample_order)
abundance_df = abundance_df[["TRAV10", "TRAV12-1"]]

In [ ]:
ax = abundance_df.plot(
    kind="bar",
    stacked=True,
    figsize=(7, 4)
)

ax.set_xlabel("sample_id")
ax.set_ylabel("Fraction of cells")
ax.set_title("TRAV10 and TRAV12-1 abundance by sample")
ax.legend(title="V gene", bbox_to_anchor=(1.05, 1), loc="upper left")

plt.tight_layout()
plt.show()